# Generate and inspect a detection dataset
Run all cells for a smoke dataset. Generation may prepare the production wallpaper assets and refuses an existing destination. Use the experiment environment as the notebook kernel.


## 1. Setup


In [ ]:
import json
import math
import os
import random
from itertools import combinations
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from tqdm import tqdm
from grid_detection.dataset.generator import GENERATOR_REVISION, generate_dataset, CAPTURE_POLICY
from grid_detection.dataset.sampling_config import SAMPLING_CONFIG
from grid_detection.dataset.yolo import YOLO_CLASSES
from grid_detection.research import DATA_DIR


## 2. Configuration
Edit the preset definitions, dataset name, seed, or workers as needed. For an existing dataset, set its name here, skip the generation cell, and continue from **Load saved dataset**.


In [ ]:
PRESET = "smoke"
DATASET_NAME = PRESET
SEED = 42
WORKERS = os.process_cpu_count() or 1

PRESETS = {
    "smoke": {"train": 20, "val": 5, "test": 5},
    "development": {"train": 2000, "val": 500, "test": 500},
}
splits = ("train", "val", "test")
counts = PRESETS[PRESET]
destination = DATA_DIR / DATASET_NAME
if GENERATOR_REVISION != 7:
    raise RuntimeError(f"Expected generator revision 7, found {GENERATOR_REVISION}")
print(f"Dataset destination: {destination}")
pprint({"preset": PRESET, "splits": counts,
        "generator_revision": GENERATOR_REVISION, "seed": SEED,
        "available_cpus": os.process_cpu_count(), "workers": WORKERS})


## 3. Sampling policy overview
These are the current production defaults; the saved configuration below is authoritative for an existing dataset. Weights are relative, and some policies are conditional.


In [ ]:
pprint({"sampling": SAMPLING_CONFIG, "capture_augmentation": CAPTURE_POLICY,
        "negative_scene_policy": "desktop with zero games"})


## 4. Generate dataset
The generator uses its production sampling, wallpaper and augmentation defaults. Existing destinations, including incomplete runs, are never overwritten. Choose a different name to generate another dataset.


In [ ]:
if destination.exists():
    raise FileExistsError(f"Dataset destination already exists: {destination}")
generate_dataset(destination, **counts, seed=SEED, workers=WORKERS, scene_details=True)
print(f"Generated dataset: {destination}")


## 5. Load saved dataset
These cells read saved files and do not depend on generation having run in this session.


In [ ]:
for name in ("data.yaml", "config.json", "manifest.jsonl", "statistics.json"):
    if not (destination / name).is_file():
        raise FileNotFoundError(destination / name)
metadata = json.loads((destination / "config.json").read_text())
statistics = json.loads((destination / "statistics.json").read_text())
with (destination / "manifest.jsonl").open() as manifest:
    records = [json.loads(line) for line in manifest if line.strip()]
print(f"Loaded dataset: {destination}")
pprint({key: metadata.get(key) for key in
        ("experiment", "generator_revision", "status", "seed", "workers", *splits)})


## 6. Verify dataset
Checks use the saved request, rather than the currently selected preset. A failed integrity check raises an error; unavailable optional diagnostics are reported separately. Empty labels are valid for negative scenes.


In [ ]:
expected = {"experiment": "grid-detection", "generator_revision": 7, "status": "complete"}
for key, value in expected.items():
    if metadata.get(key) != value:
        raise ValueError(f"Integrity failure: {key}: expected {value!r}, found {metadata.get(key)!r}")
if metadata.get("classes") != list(YOLO_CLASSES):
    raise ValueError("Integrity failure: unexpected class order")
image_stems, label_stems = {}, {}
for split in splits:
    image_stems[split] = {p.stem for p in (destination / "images" / split).glob("*.png")}
    label_stems[split] = {p.stem for p in (destination / "labels" / split).glob("*.txt")}
    requested = metadata.get(split)
    print(f"{split}: requested={requested}, images={len(image_stems[split])}, labels={len(label_stems[split])}")
    if not isinstance(requested, int) or image_stems[split] != label_stems[split] or len(image_stems[split]) != requested:
        raise ValueError(f"Integrity failure: image/label pairing or count in {split}")

identities = {split: set() for split in splits}
seen = {split: set() for split in splits}
tolerance = 1e-7
for record in tqdm(records, desc="Verifying", unit="scene"):
    split = record.get("split")
    if split not in splits:
        raise ValueError(f"Integrity failure: unknown manifest split {split!r}")
    image = destination / record["image"]
    label = destination / record["label"]
    stem = image.stem
    if (image != destination / "images" / split / f"{stem}.png"
            or label != destination / "labels" / split / f"{stem}.txt"
            or stem not in image_stems[split] or stem in seen[split]):
        raise ValueError(f"Integrity failure: invalid or duplicate manifest entry: {record}")
    seen[split].add(stem)
    identities[split].update(record["board_ids"])
    lines = label.read_text().splitlines()
    if not record["board_ids"] and lines:
        raise ValueError(f"Integrity failure: negative scene has labels: {label}")
    for row in lines:
        fields = row.split()
        if len(fields) != 5 or fields[0] not in {str(i) for i in range(len(YOLO_CLASSES))}:
            raise ValueError(f"Integrity failure: invalid YOLO row: {label}: {row}")
        x, y, w, h = map(float, fields[1:])
        if (not all(math.isfinite(v) and 0 <= v <= 1 for v in (x, y, w, h))
                or w <= 0 or h <= 0 or x-w/2 < -tolerance or y-h/2 < -tolerance
                or x+w/2 > 1+tolerance or y+h/2 > 1+tolerance):
            raise ValueError(f"Integrity failure: invalid box: {label}: {row}")
for split in splits:
    if seen[split] != image_stems[split]:
        raise ValueError(f"Integrity failure: manifest coverage in {split}")
for a, b in combinations(splits, 2):
    if identities[a] & identities[b]:
        raise ValueError(f"Integrity failure: board identities overlap: {a} / {b}")

def optional_check(label, value):
    print(f"{'UNAVAILABLE' if value is None else 'PASS' if value else 'FAIL'}: {label}")
    if value is False:
        raise ValueError(f"Integrity failure: {label}")

for split in splits:
    actual = len(seen[split])
    accepted = statistics.get("accepted", {}).get("splits", {}).get(split, {}).get("totals", {}).get("images")
    optional_check(f"{split} accepted images", accepted == actual if accepted is not None else None)
    legacy = statistics.get(split, {}).get("counts", {})
    optional_check(f"{split} statistics scenes", legacy["scenes"] == actual if "scenes" in legacy else None)
    attempt = statistics.get("attempts", {}).get("splits", {}).get(split)
    if attempt and all(k in attempt for k in ("attempted", "accepted", "rejected")):
        ok = attempt["attempted"] == attempt["accepted"] + attempt["rejected"] and attempt["accepted"] == actual
        if "retries" in legacy: ok = ok and legacy["retries"] == attempt["rejected"]
        optional_check(f"{split} attempt counts", ok)
    else:
        optional_check(f"{split} attempt counts", None)
accepted_all = statistics.get("accepted", {}).get("overall", {}).get("totals", {}).get("images")
optional_check("overall accepted images", accepted_all == len(records) if accepted_all is not None else None)
attempt_all = statistics.get("attempts", {}).get("overall")
if attempt_all and all(k in attempt_all for k in ("attempted", "accepted", "rejected")):
    optional_check("overall attempt counts", attempt_all["attempted"] == attempt_all["accepted"] + attempt_all["rejected"] and attempt_all["accepted"] == len(records))
else:
    optional_check("overall attempt counts", None)
print("PASS: identity, completion, requested counts, pairing, annotations, manifest and split isolation")


## 7. Analyze sampling distributions and performance
Configured weights, attempted scene observations, accepted images, desktops and boards have different denominators. Missing fields are excluded, not inferred. Board attempts describe observations in containing scene attempts, not independent board failures.


In [ ]:
performance = statistics.get("performance", {})
environment = metadata.get("provenance", {}).get("environment", {})
print("Saved performance (seconds and images/second):")
for label, value in {
    "accepted images": performance.get("accepted_image_count"),
    "workers": metadata.get("workers"),
    "available CPUs": environment.get("available_cpu_count"),
    "logical CPUs": environment.get("logical_cpu_count"),
    "overall generation seconds": performance.get("overall_generation_seconds"),
    "sample generation/export seconds": performance.get("sample_generation_export_seconds"),
    "asset preparation seconds": performance.get("asset_preparation_seconds"),
    "provenance collection seconds": performance.get("provenance_collection_seconds"),
    "final statistics/metadata preparation seconds": performance.get("final_statistics_metadata_preparation_seconds"),
    "overall images/second": performance.get("overall_images_per_second"),
    "sample generation images/second": performance.get("sample_generation_images_per_second"),
}.items():
    if value is None:
        displayed = "unavailable"
    elif "seconds" in label:
        displayed = f"{value:.3f}"
    elif "second" in label:
        displayed = f"{value:.2f}"
    else:
        displayed = str(value)
    print(f"  {label}: {displayed}")
print("Overall includes setup between phases; final statistics and completed configuration writes are excluded.")
for split in splits:
    summary = statistics.get(split, {})
    if summary: print(split, summary.get("counts", {}))


In [ ]:
def format_summary(value):
    """Round displayed floats while preserving counts and unavailable values."""
    if isinstance(value, float):
        return float(f"{value:.4g}")
    if isinstance(value, dict):
        return {k: format_summary(v) for k, v in value.items()}
    if isinstance(value, list):
        return [format_summary(v) for v in value]
    return value


def show_fitting(fitting):
    if not fitting:
        print("Fitting: unavailable")
        return
    print("Fitting by applicable boards (budget and placement have distinct denominators):")
    for group, buckets in fitting.items():
        for label, item in buckets.items():
            reduction = item.get("total_reduction", {})
            values = {key: item.get(key) for key in (
                "applicable_boards", "any_fitted", "any_fitted_rate",
                "budget_applicable_boards", "budget_fitted", "budget_fitting_rate",
                "placement_applicable_boards", "placement_fitted", "placement_fitting_rate")}
            values["total_reduction"] = {key: reduction.get(key) for key in ("count", "mean", "median", "p95", "max")}
            print(f"  {group}/{label}: {format_summary(values)}")

policy = metadata.get("sampling_scope", {})
accepted = statistics.get("accepted", {}).get("overall", {})
attempts = statistics.get("attempts", {}).get("overall", {})
categorical = accepted.get("categorical", {})
numerical = accepted.get("numerical", {})
print("Configured distributions below use the saved policy.")
print("Accepted totals (images, source desktops and boards):")
pprint(format_summary(accepted.get("totals", "unavailable")))

def distribution(title, values, unit):
    if not values:
        print(f"{title}: unavailable or no observations")
        return
    total = sum(values.values())
    print(f"\n{title} — {unit}, denominator={total}")
    for key, value in values.items():
        print(f"  {key}: {value} ({100 * value / total:.1f}%)" if total else f"  {key}: {value}")

def accepted_distribution(key, unit):
    distribution("Accepted " + key, categorical.get(key, {}).get("counts", {}), unit)

def attempted_distribution(key, board=False):
    source = attempts.get("boards_by_containing_attempt_outcome" if board else "by", {})
    buckets = source.get(key, {})
    distribution("Attempted " + key, {k: v.get("attempted", 0) for k, v in buckets.items()},
                 "board observations in scene attempts" if board else "scene attempts with field available")

for key in ("scene_mode", "game_count", "game_zoom"):
    distribution("Configured " + key, policy.get(key, {}), "relative weights (conditional where applicable)")
for key in ("scene_mode", "game_count", "composition", "resolution_group", "display_scale", "shell", "desktop_style"):
    attempted_distribution(key)
for key in ("game_zoom", "board_area", "sampled_cell_size"):
    attempted_distribution(key, board=True)
for key in ("scene_mode", "board_count_per_image"):
    accepted_distribution(key, "images")
for key in ("resolution_group", "display_scale", "shell", "style", "wallpaper_category", "wallpaper_subtype", "wallpaper_placement"):
    accepted_distribution("source_desktops_" + key, "source desktops with field available")
for key in ("style", "rows", "columns", "sampled_cell_size", "requested_cell_size", "resolved_cell_size", "requested_zoom"):
    accepted_distribution("board_" + key, "boards with field available")
for key in ("augmentation_state", "augmentation_blur", "augmentation_resampling", "augmentation_jpeg_quality"):
    accepted_distribution(key, "images with field available")
print("Attempt field availability:"); pprint(attempts.get("availability", "unavailable"))
print("Board field availability:"); pprint(attempts.get("board_availability", "unavailable"))


In [ ]:
print("Numerical summaries (count is the applicable image or board denominator):")
metrics = ("grid_normalized_width", "grid_normalized_height", "grid_normalized_area",
           "board_pitch_x", "board_pitch_y", "board_area_cells", "total_reduction",
           "budget_reduction", "placement_reduction", "augmentation_resize_factor",
           "augmentation_scale_x", "augmentation_scale_y", "augmentation_brightness", "augmentation_contrast")
for key in metrics:
    entry = numerical.get(key)
    print(key, format_summary({k: entry.get(k) for k in ("count", "min", "median", "mean", "p05", "p95", "max")}) if entry else "unavailable")
show_fitting(accepted.get("fitting"))
print("Attempt outcomes, recorded rejection reasons and retry frequency:")
for key in ("attempted", "accepted", "rejected", "rejection_reasons", "attempts_per_sample", "fitting_observations"):
    print(key, format_summary(attempts.get(key, "unavailable")))
print("Rejection reasons are recorded retryable outcomes; fitting markers are not rejection causes.")

plot_metrics = ("grid_normalized_area", "board_pitch_x", "total_reduction", "augmentation_resize_factor")
fig, axes = plt.subplots(2, 2, figsize=(12, 6), squeeze=False)
for ax, key in zip(axes.flat, plot_metrics):
    entry = numerical.get(key, {})
    hist = entry.get("histogram", {})
    edges, values = hist.get("edges", []), hist.get("counts", [])
    if values and len(edges) == len(values) + 1:
        ax.bar(range(len(values)), values)
        ax.set_xticks(range(len(values)), [f"{edges[i]:g}–{edges[i+1]:g}" for i in range(len(values))], rotation=45, ha="right", fontsize=7)
        ax.set_title(f"{key} (n={entry.get('count')}; under={hist.get('underflow')}, over={hist.get('overflow')})")
    else:
        ax.text(.5, .5, "Unavailable", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(key)
    ax.set_ylabel("observations")
fig.tight_layout()
plt.show()


## 8. Visualize dataset samples
Reproducible random samples and representative categories use the saved images and all saved YOLO annotations. Categories missing from a small dataset are reported; selected images are deduplicated.


In [ ]:
rng = random.Random(SEED)
selected = {}
def choose(label, record):
    if record is None:
        print(f"Unavailable: {label}")
    else:
        selected.setdefault(record["image"], {"record": record, "reasons": []})["reasons"].append(label)

def box_areas(record):
    size = record.get("final_size")
    if not size or not size[0] or not size[1]: return []
    return [(b["final_grid_box"][2]-b["final_grid_box"][0]) *
            (b["final_grid_box"][3]-b["final_grid_box"][1]) / (size[0]*size[1])
            for b in record.get("boards", []) if b.get("final_grid_box")]

def fitted(record):
    return any(b.get("sampled_cell_size") is not None and b.get("resolved_cell_size") is not None
               and b["sampled_cell_size"] > b["resolved_cell_size"] for b in record.get("boards", []))

for record in rng.sample(records, min(3, len(records))): choose("random", record)
for mode in ("desktop", "game", "grid"):
    choose(mode, next((r for r in records if r.get("scene_mode") == mode), None))
choose("negative", next((r for r in records if not r.get("board_ids")), None))
choose("multi-board", next((r for r in records if len(r.get("board_ids", [])) > 1), None))
choose("fitted board", next((r for r in records if fitted(r)), None))
scaled = [r for r in records if (r.get("desktop") or {}).get("screen", {}).get("scale") is not None]
choose("highest desktop scale", max(scaled, key=lambda r: r["desktop"]["screen"]["scale"]) if scaled else None)
with_boxes = [r for r in records if box_areas(r)]
choose("smallest grid box", min(with_boxes, key=lambda r: min(box_areas(r))) if with_boxes else None)
choose("largest grid box", max(with_boxes, key=lambda r: max(box_areas(r))) if with_boxes else None)

items = list(selected.values())
for start in range(0, len(items), 4):
    batch = items[start:start+4]
    fig, axes = plt.subplots(1, len(batch), figsize=(5*len(batch), 4), squeeze=False)
    for ax, item in zip(axes.flat, batch):
        record = item["record"]
        with Image.open(destination / record["image"]) as image:
            ax.imshow(image)
            width, height = image.size
        classes_shown = set()
        for line in (destination / record["label"]).read_text().splitlines():
            class_id, x, y, w, h = map(float, line.split())
            class_id = int(class_id)
            left, top = (x-w/2)*width, (y-h/2)*height
            color = f"C{class_id}"
            ax.add_patch(Rectangle((left, top), w*width, h*height, fill=False, edgecolor=color, linewidth=1.5))
            classes_shown.add(class_id)
        if classes_shown:
            from matplotlib.patches import Patch
            ax.legend(handles=[Patch(facecolor="none", edgecolor=f"C{i}", label=YOLO_CLASSES[i])
                               for i in sorted(classes_shown)], fontsize=6, loc="lower right")
        scale = (record.get("desktop") or {}).get("screen", {}).get("scale")
        scale_text = f" | desktop scale={scale:g}" if scale is not None else ""
        ax.set_title(f"{', '.join(item['reasons'])}\n{record['split']}/{Path(record['image']).name} | {record.get('scene_mode')} | boards={len(record.get('board_ids', []))}{scale_text}", fontsize=8)
        ax.axis("off")
    fig.tight_layout()
    plt.show()
